# Simulating the Wealth Distribution

You now build artificial economies of 10,000 adults. In Sections 2.1 and 2.2, keep each simulated adult's age group fixed and change how you assign wealth. Put the simulation in a documented function in a separate module and import it into your notebook. Let the function take the age shares as an argument: you will change them in Section 2.4.

# Problem 2.1: Give Everybody Their Age-Group Median

1. Divide each age count by the total across the eight age groups. These shares are the probabilities of belonging to each group. Check that they sum to one.
2. Draw an age group for each adult. Draw a number $u$ from the uniform distribution on $[0, 1)$. Split $[0, 1)$ into eight intervals whose lengths equal the age shares: the first group covers $[0, s_1)$, the second $[s_1, s_1 + s_2)$, and so on. The adult belongs to the group whose interval contains $u$.
3. Give each simulated adult the P50 wealth of their group. Compare the simulated age shares with the published shares. Why do they differ slightly?

*Hint.* Use `rng = np.random.default_rng(42)` and `rng.uniform(0, 1, size=10000)` as in Lecture 8. Loop over the eight groups, keep a running sum of the shares for the interval bounds, and select adults with a Boolean mask such as `(u >= lower) & (u < upper)`. `np.mean(age == j)` is the simulated share of group j. Keep the shares and the wealth data in the same age order.

## Problem 2.1.1

Divide each age count by the total across the eight age groups. These shares are the probabilities of belonging to each group. Check that they sum to one.

In [1]:
from io import BytesIO
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(table, **selectors):
    """Download a StatBank table as a DataFrame. Each keyword is one variable code; '*' selects all values."""
    url = f'https://api.statbank.dk/v1/data/{table}/CSV'
    params = dict(lang='en', valuePresentation='Code', **selectors)
    response = requests.get(url, params=params, timeout=45)
    response.raise_for_status()
    frame = pd.read_csv(BytesIO(response.content), sep=';', dtype=str, encoding='utf-8-sig')
    frame = frame.rename(columns={frame.columns[-1]: 'value'})
    frame['value'] = pd.to_numeric(frame['value'].str.replace(',', '.', regex=False))
    return frame

settings = dict(PRISENHED='006', ALDER='*', Tid='2024')  # nominal DKK, all age groups, 2024
percentiles = download('FORMUE16', PERCENTIL='*', **settings)
wealth = download('FORMUE15', FORMUE='*', DEFINITION='D20', **settings)

age_labels = {
    '1802': 'All adults',
    '1829': '18-29 Years',
    '3039': '30-39 Years',
    '4049': '40-49 Years',
    '5059': '50-59 Years',
    '6069': '60-69 Years',
    '7079': '70-79 Years',
    '8089': '80-89 Years',
    '9099': '90+'
}
age_order = ['1829', '3039', '4049', '5059', '6069', '7079', '8089', '9099']

# Total number of adults in each of the eight age groups
age_counts = (
    wealth.loc[wealth['ALDER'].isin(age_order)]
    .groupby('ALDER')['value']
    .sum()
    .reindex(age_order)
)

# Age shares used as interval lengths and probabilities
age_shares = (age_counts / age_counts.sum()).to_numpy()

age_share_table = pd.DataFrame({
    'Age': [age_labels[code] for code in age_order],
    'Count': age_counts.to_numpy(),
    'Share': age_shares,
})

print(f'Sum of age shares: {age_shares.sum():.12f}')
print(f'Check (sum == 1):  {np.isclose(age_shares.sum(), 1.0)}')
age_share_table.style.hide(axis='index').format({
    'Count': '{:,.0f}',
    'Share': '{:.4f}',
})

Sum of age shares: 1.000000000000
Check (sum == 1):  True


Age,Count,Share
18-29 Years,"927,354",0.1911
30-39 Years,"783,901",0.1616
40-49 Years,"708,080",0.1459
50-59 Years,"802,113",0.1653
60-69 Years,"709,729",0.1463
70-79 Years,"581,304",0.1198
80-89 Years,"291,975",0.0602
90+,"47,362",0.0098


> Explanation 2.1.1:

The eight age counts are found by summing all 16 FORMUE15 wealth brackets within each age group. They add up to 4,851,818 adults, so dividing each count by this total gives the eight age shares. The shares sum to one by construction; the printout checks this to machine precision. The largest group is 18-29 years (19.1% of adults) and the smallest is 90+ years (0.98%). The order `age_order` is used for both the counts and the rows of the percentile table, so the shares and wealth thresholds always refer to the same age group.

## Problem 2.1.2

Draw an age group for each adult. Draw $u$ from the uniform distribution on $[0, 1)$, split $[0, 1)$ into eight intervals whose lengths equal the age shares, and place each adult in the group whose interval contains $u$.

In [2]:
from Simulation import simulate_wealth

# Percentiles with age groups in rows and P1 to P99 in columns, sorted as numbers
age_percentiles = percentiles.loc[percentiles['ALDER'].isin(age_order)].copy()
age_percentiles['rank'] = (
    age_percentiles['PERCENTIL'].str.replace('P', '', regex=False).astype(int)
)
thresholds_df = (
    age_percentiles
    .pivot(index='ALDER', columns='rank', values='value')
    .reindex(age_order)
)
thresholds = thresholds_df.to_numpy()

print(f'Threshold matrix shape: {thresholds.shape}')
print('Columns run from P1 to P99:', list(thresholds_df.columns[:3]), '...', list(thresholds_df.columns[-3:]))

# Draw an age group for every adult. The second return value is the group P50,
# which Problem 2.1.3 uses.
age, wealth_medians = simulate_wealth(
    age_shares, thresholds, n=10_000, seed=42, within_group=False
)

simulated_counts = np.bincount(age, minlength=len(age_order))
print(f'All {age.size:,} adults have a valid age group: '
      f'{np.all((age >= 0) & (age < len(age_order)))}')

pd.DataFrame({
    'Age': [age_labels[code] for code in age_order],
    'Simulated count': simulated_counts,
}).style.hide(axis='index').format({'Simulated count': '{:,.0f}'})

Threshold matrix shape: (8, 99)
Columns run from P1 to P99: [1, 2, 3] ... [97, 98, 99]
All 10,000 adults have a valid age group: True


Age,Simulated count
18-29 Years,"1,942"
30-39 Years,"1,624"
40-49 Years,"1,438"
50-59 Years,"1,703"
60-69 Years,"1,426"
70-79 Years,"1,179"
80-89 Years,578
90+,110


> Explanation 2.1.2:

The function draws one uniform number $u$ per adult. It walks through the eight shares, keeps a running lower bound, and places adult $i$ in group $j$ when $lower \le u_i < lower + s_j$; the last interval takes every remaining draw (`u >= lower`), so every adult is assigned. Because the draws are random, the simulated counts are close to $10{,}000 \times s_j$ but not equal to them: the expected count is $10{,}000 s_j$, while the realised count varies around that expectation. The same seed and the same age shares are used again in 2.2, so both economies share exactly the same simulated ages and differ only in how wealth is assigned within each group.

## Problem 2.1.3

Give each simulated adult the P50 wealth of their group. Compare the simulated age shares with the published shares. Why do they differ slightly?

In [3]:
simulated_shares = np.array([np.mean(age == j) for j in range(len(age_order))])

share_comparison = pd.DataFrame({
    'Age': [age_labels[code] for code in age_order],
    'Published share': age_shares,
    'Simulated share': simulated_shares,
    'Difference (pp)': 100 * (simulated_shares - age_shares),
})

print(f'Simulated shares sum: {simulated_shares.sum():.12f}')
print(f'Group medians (simulated wealth) range from '
      f'{wealth_medians.min():,.0f} to {wealth_medians.max():,.0f} DKK')
share_comparison.style.hide(axis='index').format({
    'Published share': '{:.4f}',
    'Simulated share': '{:.4f}',
    'Difference (pp)': '{:+.2f}',
})

Simulated shares sum: 1.000000000000
Group medians (simulated wealth) range from 68,806 to 2,166,898 DKK


Age,Published share,Simulated share,Difference (pp)
18-29 Years,0.1911,0.1942,+0.31
30-39 Years,0.1616,0.1624,+0.08
40-49 Years,0.1459,0.1438,-0.21
50-59 Years,0.1653,0.1703,+0.50
60-69 Years,0.1463,0.1426,-0.37
70-79 Years,0.1198,0.1179,-0.19
80-89 Years,0.0602,0.0578,-0.24
90+,0.0098,0.0110,+0.12


> Explanation 2.1.3:

The simulated shares are close to the published shares but not identical. The largest difference is in the 50-59 group, which is 0.50 percentage points too large in the simulation, while the 40-49 and 60-69 groups are about 0.21 and 0.37 percentage points too small. The reason is random sampling: each adult's uniform draw $u$ is random, so a finite sample of 10,000 adults does not reproduce the population shares exactly. The expected share of group $j$ is still the published share, and the differences would shrink if we increased the number of simulated adults. In the first economy every adult in a group receives the same P50, so all simulated wealth values are positive.

# Problem 2.2: Allow Wealth to Vary Within Each Age Group

Keep each person's age group. Draw a rank from 1 to 99 for each adult, with each rank equally likely. Assign the published wealth threshold for that person's age group and rank. For example, a person aged 40-49 who draws rank 25 receives P25 for ages 40-49.

*Hint.* `rng.integers(1, 100, size=10000)` draws the ranks (Lecture 8); draw them after the ages. Store the percentiles in a NumPy array with age groups in rows and P1 to P99 in columns, sorted as numbers so that P2 comes before P10. P1 is then column 0. A loop over the adults with `values[a, r - 1]` works.

## Problem 2.2.1

Keep each person's age group. Draw a rank from 1 to 99 for each adult, with each rank equally likely, and assign the published wealth threshold for that person's age group and rank.

In [4]:
# Economy 2: keep each adult's age group, but draw a rank from 1 to 99
age_within, wealth_within = simulate_wealth(
    age_shares, thresholds, n=10_000, seed=42, within_group=True
)

# The same seed and the same age shares give the same simulated ages as in 2.1
print(f'Same age groups as the median economy: {np.array_equal(age_within, age)}')

# Show the range of wealth values inside a few age groups
within_ranges = pd.DataFrame({
    'Age': [age_labels[code] for code in age_order],
    'Distinct values': [
        len(np.unique(wealth_within[age_within == j]))
        for j in range(len(age_order))
    ],
    'Minimum': [
        wealth_within[age_within == j].min()
        for j in range(len(age_order))
    ],
    'Maximum': [
        wealth_within[age_within == j].max()
        for j in range(len(age_order))
    ],
})
within_ranges.style.hide(axis='index').format({
    'Distinct values': '{:,.0f}',
    'Minimum': '{:,.0f}',
    'Maximum': '{:,.0f}',
})

Same age groups as the median economy: True


Age,Distinct values,Minimum,Maximum
18-29 Years,99,"-437,821","2,565,268"
30-39 Years,99,"-1,097,902","6,262,472"
40-49 Years,99,"-1,128,378","15,146,012"
50-59 Years,99,"-968,947","27,671,225"
60-69 Years,99,"-569,677","28,648,916"
70-79 Years,99,"-372,779","24,836,677"
80-89 Years,99,"-357,473","21,530,359"
90+,74,"-131,687","15,594,617"


> Explanation 2.2.1:

The second economy keeps each adult's age group but replaces the common group median with a randomly drawn rank from 1 to 99. Each rank is equally likely, so rank $r$ is assigned the published P$r$ threshold in column $r-1$ of `thresholds`. Within each age group wealth therefore takes many different values, ranging from P1 to P99, instead of only the median. The same seed and the same age shares reproduce exactly the same simulated ages as in 2.1; the rank draws then create the within-group variation. This is a discrete approximation to each age group's full wealth distribution: each of the 99 published percentiles receives about 1% of the simulated adults in that group. Because the lower percentiles are negative in the youngest groups, this economy also produces people with negative net wealth.

# Problem 2.3: Compare Both Economies with Denmark

Make one table with three rows: observed Denmark, age-group medians only, and variation within age groups. Report P10, P50, P90, P90 - P10 (all in DKK), and the percentage with negative wealth. For Denmark, take P10, P50 and P90 from the all-adult row of FORMUE16 and the negative-wealth percentage from the all-adult rows of FORMUE15.

1. What fraction of the observed P90 - P10 gap does the first economy reproduce? Why does it contain no one with negative wealth?
2. The second economy uses the age-specific percentiles directly. How close should it come to Denmark? Give one reason why a difference remains.

*Hint.* `np.percentile(wealth, 10)` works like `np.median`, with the percentile as the second argument. `100*np.mean(wealth < 0)` is the percentage below zero.

## Problem 2.3.1

Make the three-row table. What fraction of the observed P90 - P10 gap does the first economy reproduce? Why does it contain no one with negative wealth?

In [5]:
# The three FORMUE15 brackets that lie below zero
negative_brackets = ['0000M10', 'M100M05', 'M050000']

# All-adult percentiles from FORMUE16
all_adult_percentiles = (
    percentiles.loc[percentiles['ALDER'] == '1802']
    .assign(rank=lambda d: d['PERCENTIL'].str.replace('P', '', regex=False).astype(int))
    .set_index('rank')['value']
)

# All-adult negative-wealth share from FORMUE15
all_adult_counts = wealth.loc[wealth['ALDER'] == '1802']
negative_share_denmark = 100 * (
    all_adult_counts.loc[all_adult_counts['FORMUE'].isin(negative_brackets), 'value'].sum()
    / all_adult_counts['value'].sum()
)

def describe(wealth_values):
    """Return the five requested statistics for a simulated wealth vector."""
    p10, p50, p90 = np.percentile(wealth_values, [10, 50, 90])
    return {
        'P10': p10,
        'P50': p50,
        'P90': p90,
        'P90 - P10': p90 - p10,
        'Negative wealth (%)': 100 * np.mean(wealth_values < 0),
    }

def format_stat(value):
    """Format DKK amounts without decimals and percentages with two decimals."""
    if abs(value) >= 1_000 or float(value).is_integer():
        return f'{value:,.0f}'
    return f'{value:,.2f}'

def format_change(value):
    """Format a difference, keeping two decimals for percentage-point changes."""
    if abs(value) >= 1_000 or float(value).is_integer():
        return f'{value:+,.0f}'
    return f'{value:+.2f}'

comparison = pd.DataFrame([
    {
        'Economy': 'Observed Denmark',
        'P10': all_adult_percentiles[10],
        'P50': all_adult_percentiles[50],
        'P90': all_adult_percentiles[90],
        'P90 - P10': all_adult_percentiles[90] - all_adult_percentiles[10],
        'Negative wealth (%)': negative_share_denmark,
    },
    {'Economy': 'Age-group medians only', **describe(wealth_medians)},
    {'Economy': 'Variation within age groups', **describe(wealth_within)},
]).set_index('Economy')

gap_fraction = (
    comparison.loc['Age-group medians only', 'P90 - P10']
    / comparison.loc['Observed Denmark', 'P90 - P10']
)
print(f'First economy reproduces {100 * gap_fraction:.1f}% of the observed P90 - P10 gap.')
comparison.style.format({
    'P10': '{:,.0f}',
    'P50': '{:,.0f}',
    'P90': '{:,.0f}',
    'P90 - P10': '{:,.0f}',
    'Negative wealth (%)': '{:.2f}',
})

First economy reproduces 46.7% of the observed P90 - P10 gap.


,P10,P50,P90,P90 - P10,Negative wealth (%)
Economy,,,,,
Observed Denmark,"4,875","813,665","4,501,923","4,497,048",7.99
Age-group medians only,"68,806","1,024,656","2,166,898","2,098,092",0.00
Variation within age groups,"8,639","813,339","4,377,805","4,369,166",7.54


> Explanation 2.3.1:

The first economy reproduces **46.7%** of Denmark's observed P90 - P10 gap (2,098,092 DKK out of 4,497,048 DKK). It contains no one with negative wealth because it gives every adult their age group's median, and all eight group medians are positive; the smallest is 68,806 DKK for 18-29-year-olds. By construction it removes both tails of each group's distribution, so it understates P10 badly (68,806 versus 4,875 DKK) and also understates P90 (2,166,898 versus 4,501,923 DKK).

## Problem 2.3.2

The second economy uses the age-specific percentiles directly. How close should it come to Denmark? Give one reason why a difference remains.

In [6]:
denmark = comparison.loc['Observed Denmark']
within = comparison.loc['Variation within age groups']

closeness = pd.DataFrame({
    'Observed Denmark': denmark,
    'Variation within age groups': within,
    'Difference': within - denmark,
    'Relative difference (%)': 100 * (within / denmark - 1),
})
closeness.style.format({
    'Observed Denmark': format_stat,
    'Variation within age groups': format_stat,
    'Difference': format_change,
    'Relative difference (%)': lambda value: f'{value:+.2f}',
})

,Observed Denmark,Variation within age groups,Difference,Relative difference (%)
P10,"4,875","8,639","+3,764",+77.20
P50,"813,665","813,339",-326,-0.04
P90,"4,501,923","4,377,805","-124,118",-2.76
P90 - P10,"4,497,048","4,369,166","-127,882",-2.84
Negative wealth (%),7.99,7.54,-0.45,-5.62


> Explanation 2.3.2:

The second economy comes much closer to Denmark. Its median is 813,339 DKK against 813,665 DKK, a difference of only 326 DKK (0.04%). P10 is 8,639 versus 4,875 DKK, P90 is 4,377,805 versus 4,501,923 DKK, the P90 - P10 gap is 4,369,166 versus 4,497,048 DKK (about 2.8% lower), and the negative-wealth share is 7.54% versus 7.99% (0.45 percentage points lower). It should be close because it uses the same age shares and the published within-group percentile thresholds directly. A difference remains because the simulation is a finite random sample of 10,000 adults, so its age mix differs slightly from the population (see 2.1.3), and because 99 discrete, rounded percentile points are only an approximation to each group's continuous wealth distribution.

# Problem 2.4: An Older Denmark

Keep the age-specific wealth distributions from 2.2, but change the age shares: move 5 percentage points of the adult population from the 18-29 group to the 70-79 group. Run the second economy again with the same seed and report the same five statistics next to your results from 2.2.

1. Does the older population have a higher median? A wider or narrower P90 - P10 gap? More or fewer people with negative wealth? Use your age profiles from Part 1 to explain each answer.
2. Why is it useful to use the same seed in both runs?
3. This experiment holds the 2024 age profiles fixed. Give one reason why they might change if the population really aged.

*Hint.* If your function takes the age shares as an argument, the new run takes one new line of code.

## Problem 2.4.1

Run the second economy again with the older age shares and the same seed. Report the same five statistics next to your results from 2.2. Does the older population have a higher median? A wider or narrower P90 - P10 gap? More or fewer people with negative wealth? Use your age profiles from Part 1 to explain each answer.

In [7]:
# Move 5 percentage points from 18-29 to 70-79
older_shares = age_shares.copy()
older_shares[0] -= 0.05  # 18-29 years
older_shares[5] += 0.05  # 70-79 years

print(f'Sum of older shares: {older_shares.sum():.12f}')

# Run the second economy again with the same seed
age_older, wealth_older = simulate_wealth(
    older_shares, thresholds, n=10_000, seed=42, within_group=True
)

older_comparison = pd.DataFrame({
    '2024 age shares (2.2)': describe(wealth_within),
    'Older population (2.4)': describe(wealth_older),
})
older_comparison['Difference'] = (
    older_comparison['Older population (2.4)']
    - older_comparison['2024 age shares (2.2)']
)

older_comparison.style.format({
    '2024 age shares (2.2)': format_stat,
    'Older population (2.4)': format_stat,
    'Difference': format_change,
})

Sum of older shares: 1.000000000000


,2024 age shares (2.2),Older population (2.4),Difference
P10,"8,639","12,894","+4,255"
P50,"813,339","944,718","+131,379"
P90,"4,377,805","4,657,071","+279,266"
P90 - P10,"4,369,166","4,644,177","+275,010"
Negative wealth (%),7.54,7.10,-0.44


> Explanation 2.4.1:

Moving 5 percentage points from 18-29 to 70-79 lowers the 18-29 share from 19.1% to 14.1% and raises the 70-79 share from 12.0% to 17.0%. With the same seed, the older economy has a **higher median** (944,718 versus 813,339 DKK), a **wider P90 - P10 gap** (4,644,177 versus 4,369,166 DKK), and **fewer people with negative wealth** (7.10% versus 7.54%).

These results follow directly from the Part 1 age profiles. Median wealth is only 68,806 DKK at 18-29 but 1,733,634 DKK at 70-79, so replacing young adults by older adults raises the median. The 70-79 group also has a much wider spread (P90 - P10 = 6,437,553 - 174,039 = 6,263,514 DKK) than the 18-29 group (541,878 - (-14,254) = 556,132 DKK), and its upper tail is far higher (P90 of 6.44 million versus 0.54 million DKK), so the overall gap widens even though its P10 is also higher. Finally, the negative-wealth share is 3.0% at 70-79 but 12.2% at 18-29, so shifting population away from the youngest group reduces the overall share with negative wealth.

## Problem 2.4.2

Why is it useful to use the same seed in both runs?

In [8]:
# Rerun the older economy with a different seed to show the sampling noise
_, wealth_older_alt = simulate_wealth(
    older_shares, thresholds, n=10_000, seed=7, within_group=True
)

seed_comparison = pd.DataFrame({
    'Same seed (42)': describe(wealth_older),
    'Different seed (7)': describe(wealth_older_alt),
})
seed_comparison['Difference'] = (
    seed_comparison['Different seed (7)'] - seed_comparison['Same seed (42)']
)

seed_comparison.style.format({
    'Same seed (42)': format_stat,
    'Different seed (7)': format_stat,
    'Difference': format_change,
})

,Same seed (42),Different seed (7),Difference
P10,"12,894","14,361","+1,467"
P50,"944,718","962,399","+17,681"
P90,"4,657,071","4,533,275","-123,796"
P90 - P10,"4,644,177","4,518,914","-125,263"
Negative wealth (%),7.10,7.17,+0.07


> Explanation 2.4.2:

Using the same seed is useful because it keeps the random draws fixed: both runs use the same sequence of uniform draws and the same rank draws. The age boundaries move, so the simulated ages are not identical, but common random numbers remove most Monte Carlo noise and make the comparison reflect the change in age shares rather than luck in the sampling. The rerun with seed 7 shows the size of that noise: changing only the seed moves the median by 17,681 DKK and the P90 - P10 gap by 125,263 DKK. These are 13% and 46% of the corresponding changes from 2.2 to 2.4, so without a common seed the sampling noise would obscure a substantial part of the demographic effect.

## Problem 2.4.3

This experiment holds the 2024 age profiles fixed. Give one reason why they might change if the population really aged.

In [9]:
# Only the number of adults in each age group changes; the age profiles are fixed
composition = pd.DataFrame({
    'Age': [age_labels[code] for code in age_order],
    '2024 adults (of 10,000)': 10_000 * age_shares,
    'Older adults (of 10,000)': 10_000 * older_shares,
    'Change': 10_000 * (older_shares - age_shares),
})
composition.style.hide(axis='index').format({
    '2024 adults (of 10,000)': '{:,.0f}',
    'Older adults (of 10,000)': '{:,.0f}',
    'Change': '{:+,.0f}',
})

Age,"2024 adults (of 10,000)","Older adults (of 10,000)",Change
18-29 Years,"1,911","1,411",-500
30-39 Years,"1,616","1,616",+0
40-49 Years,"1,459","1,459",+0
50-59 Years,"1,653","1,653",+0
60-69 Years,"1,463","1,463",+0
70-79 Years,"1,198","1,698",+500
80-89 Years,602,602,+0
90+,98,98,+0


> Explanation 2.4.3:

The experiment holds the 2024 age profiles fixed, which would not be true if the population really aged. The future 70-79-year-olds will be drawn from today's younger cohorts, and they may enter old age with different housing wealth, mortgage debt, pension entitlements and saving behaviour. House prices, interest rates, pension returns, taxes and the retirement age could also change, shifting the whole age-specific wealth distribution rather than only the number of people in each age group. The counterfactual therefore isolates the effect of the age composition and should not be read as a forecast of wealth in an older Denmark.